In [1]:
from dask_setup import setup_dask_client 

from pathlib import Path
import glob
from zarr.codecs import BloscCodec

import xarray as xr

In [2]:
client, cluster, dask_tmp = setup_dask_client(mode="interactive", workload_type="cpu")   # heavy compute

INFO     [client] Interactive cluster mode — using already-allocated nodes
INFO     [multinode] Interactive cluster: single node, using LocalCluster (node=gadi-cpu-spr-0330.gadi.nci.org.au)
INFO     [client] Starting Dask client setup (workload_type=cpu | environment=jupyter)
INFO     [resources] Resources detected via PBS (total_cores=52 | total_mem_gib=248.0)


INFO     [client] Temp/spill dir: /jobfs/178024193.gadi-pbs/dask-1931345
INFO     [client] Workers: 52 | threads/worker: 1 | processes: True
INFO     [client] Mem: total ~248.0 GiB | usable ~198.0 GiB | per-worker ~3.8 GiB
INFO     [client] Compression: spill=auto | comm=False
INFO     [client] Dask client ready


[setup_dask_client] Configuration summary
temp/spill dir: /jobfs/178024193.gadi-pbs/dask-1931345
Workers: 52 | threads/worker: 1 | processes: True
Memory: total ~248.0 GiB | usable ~198.0 GiB | per-worker ~3.8 GiB
Compression: spill=auto | comm=False


2026-09-02 14:30:14,147 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2f654a87cd7689b737bfe5ece85c68da initialized by task ('rechunk-merge-rechunk-transfer-a86d802ff65232a3f0851baea20dac57', 0, 0, 0, 9, 0, 0) executed on worker tcp://127.0.0.1:33277
2026-09-02 14:30:43,597 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 2f654a87cd7689b737bfe5ece85c68da deactivated due to stimulus 'task-finished-1788323443.5320508'


In [3]:
%cd /g/data/w42/dr6273/work/wind_drought/
import functions as fn

%load_ext autoreload
%autoreload 2

/g/data/w42/dr6273/work/wind_drought


In [4]:
ERA5_PATH = "/g/data/su28/ERA5/daily/"
ERA5_WRITE_PATH = "/g/data/ng72/dr6273/work/projects/wind_drought/data/ERA5/"

### ERA5 data

In [5]:
def get_files(path, var, start=1979, end=2025):
    """ Return files between start and end month """
    files = sorted(Path(path).glob(
        var + "/*.nc"
    ))
    
    files = [
        f for f in files
        if start <= int(f.stem[-4:]) <= end
    ]
    
    return files

In [6]:
def z500_preprocess(ds):
    """ Return 500 hPa level and smaller region """
    ds = ds.sel(
        lon=slice(REGION[0], REGION[1]),
        lat=slice(REGION[3], REGION[2]),
        level=500
    )
    return ds

In [7]:
def u300_preprocess(ds):
    """ Return 300 hPa level and smaller region """
    ds = ds.sel(
        lon=slice(REGION[0], REGION[1]),
        lat=slice(REGION[3], REGION[2]),
        level=300
    )
    return ds

In [8]:
def open_era5(files, preprocess):
    """
    Open multiple files and preprocess to region.
    """
    ds = xr.open_mfdataset(
        files,
        preprocess=preprocess,
        chunks='auto',
        parallel=True
    )
    return ds

In [9]:
def doy_anom(ds):
    """ Day of year anomalies """
    return ds.groupby('time.dayofyear') - ds.groupby('time.dayofyear').mean()

In [10]:
def specify_encoding(ds):
    """ Return encoding dict """
    codec = BloscCodec(cname='zstd', clevel=6, shuffle='bitshuffle')

    encoding = {}
    for var in ds.data_vars:
        chunks = tuple(ds[var].chunksizes[dim][0] for dim in ds[var].dims)
        
        encoding[var] = {
            'chunks': chunks, #(366, 319, 5),
            'compressors': codec,
            'dtype': 'float64'
        }
    return encoding

In [11]:
def write_zarr(ds, path, filename, encoding):
    """ Write to zarr """
    ds.to_zarr(
        path + filename,
        mode='w',
        encoding=encoding,
        zarr_format=3,
        consolidated=False
    )

In [12]:
REGION = [None, None, 0, None] # Southern hemisphere
YEARS = range(1979, 2026)

z500

In [64]:
z500_files = get_files(ERA5_PATH, 'z', start=YEARS[0], end=YEARS[-1])

In [65]:
z500 = open_era5(
    z500_files,
    preprocess=z500_preprocess
).drop_vars('time_bnds')

In [67]:
z500

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 16802, lat: 90, lon: 360)
Coordinates:
  * time     (time) datetime64[ns] 134kB 1979-01-01T11:00:00 ... 2024-12-31T1...
  * lat      (lat) float64 720B -89.5 -88.5 -87.5 -86.5 ... -3.5 -2.5 -1.5 -0.5
  * lon      (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
    level    int32 4B 500
Data variables:
    z        (time, lat, lon) float32 2GB dask.array<chunksize=(65, 65, 360), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.4.3 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Mon May 12 23:59:29 2025: cdo mergetime /g/data/if69/ls3248...
    license:      Licence to use Copernicus Products: https://apps.ecmwf.int/...
    summary:      ERA5 is the fifth generation ECMWF atmospheric reanalysis o...
    title:        ERA5 pressure-levels oper geopotential 19790101-19790131
    frequency:    day
    CDO:          Climate Data Operators version 2.4.3 (https://mpimet.mpg.de...

In [73]:
z500_chunked = z500.chunk({"time": -1, "lat": -1, "lon": 30})

In [74]:
z500_chunked

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 16802, lat: 90, lon: 360)
Coordinates:
  * time     (time) datetime64[ns] 134kB 1979-01-01T11:00:00 ... 2024-12-31T1...
  * lat      (lat) float64 720B -89.5 -88.5 -87.5 -86.5 ... -3.5 -2.5 -1.5 -0.5
  * lon      (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
    level    int32 4B 500
Data variables:
    z        (time, lat, lon) float32 2GB dask.array<chunksize=(16802, 90, 30), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.4.3 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Mon May 12 23:59:29 2025: cdo mergetime /g/data/if69/ls3248...
    license:      Licence to use Copernicus Products: https://apps.ecmwf.int/...
    summary:      ERA5 is the fifth generation ECMWF atmospheric reanalysis o...
    title:        ERA5 pressure-levels oper geopotential 19790101-19790131
    frequency:    day
    CDO:          Climate Data Operators version 2.4.3 (https://mpimet.mpg.de...

In [75]:
z500_anoms = doy_anom(z500_chunked)

In [76]:
z500_anoms

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 16802, lon: 360, lat: 90)
Coordinates:
  * time       (time) datetime64[ns] 134kB 1979-01-01T11:00:00 ... 2024-12-31...
    level      (time) int32 67kB 500 500 500 500 500 500 ... 500 500 500 500 500
    dayofyear  (time) int64 134kB 1 2 3 4 5 6 7 ... 360 361 362 363 364 365 366
  * lon        (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
  * lat        (lat) float64 720B -89.5 -88.5 -87.5 -86.5 ... -2.5 -1.5 -0.5
Data variables:
    z          (time, lat, lon) float32 2GB dask.array<chunksize=(366, 90, 30), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.4.3 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Mon May 12 23:59:29 2025: cdo mergetime /g/data/if69/ls3248...
    license:      Licence to use Copernicus Products: https://apps.ecmwf.int/...
    summary:      ERA5 is the fifth generation ECMWF atmospheric reanalysis o...
    title:        ERA5 pressure-levels oper geopotential 19790101-19790131
    frequency:    day
    CDO:          Climate Data Operators version 2.4.3 (https://mpimet.mpg.de...

In [77]:
z500_anoms = z500_anoms.chunk({"time": -1, "lat": -1, "lon": 30})

In [78]:
z500_anoms

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 16802, lon: 360, lat: 90)
Coordinates:
  * time       (time) datetime64[ns] 134kB 1979-01-01T11:00:00 ... 2024-12-31...
    level      (time) int32 67kB dask.array<chunksize=(16802,), meta=np.ndarray>
    dayofyear  (time) int64 134kB dask.array<chunksize=(16802,), meta=np.ndarray>
  * lon        (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
  * lat        (lat) float64 720B -89.5 -88.5 -87.5 -86.5 ... -2.5 -1.5 -0.5
Data variables:
    z          (time, lat, lon) float32 2GB dask.array<chunksize=(16802, 90, 30), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.4.3 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Mon May 12 23:59:29 2025: cdo mergetime /g/data/if69/ls3248...
    license:      Licence to use Copernicus Products: https://apps.ecmwf.int/...
    summary:      ERA5 is the fifth generation ECMWF atmospheric reanalysis o...
    title:        ERA5 pressure-levels oper geopotential 19790101-19790131
    frequency:    day
    CDO:          Climate Data Operators version 2.4.3 (https://mpimet.mpg.de...

In [103]:
z500_encoding = specify_encoding(z500_anoms)

In [104]:
z500_encoding['z']

{'chunks': (16802, 90, 30),
 'compressors': BloscCodec(_tunable_attrs={'typesize'}, typesize=1, cname=<BloscCname.zstd: 'zstd'>, clevel=6, shuffle=<BloscShuffle.bitshuffle: 'bitshuffle'>, blocksize=0),
 'dtype': 'float64'}

In [105]:
write_zarr(
    z500_anoms,
    ERA5_WRITE_PATH,
    'z500_dayofyear_anoms_1979-2024.zarr',
    z500_encoding
)

u300

In [13]:
u300_files = get_files(ERA5_PATH, 'u', start=YEARS[0], end=YEARS[-1])

In [14]:
u300 = open_era5(
    u300_files,
    preprocess=u300_preprocess
).drop_vars('time_bnds')

In [15]:
u300

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 16802, lat: 90, lon: 360)
Coordinates:
  * time     (time) datetime64[ns] 134kB 1979-01-01T11:00:00 ... 2024-12-31T1...
  * lat      (lat) float64 720B -89.5 -88.5 -87.5 -86.5 ... -3.5 -2.5 -1.5 -0.5
  * lon      (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
    level    int32 4B 300
Data variables:
    u        (time, lat, lon) float32 2GB dask.array<chunksize=(65, 65, 360), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.4.3 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Tue May 13 00:44:27 2025: cdo mergetime /g/data/if69/ls3248...
    license:      Licence to use Copernicus Products: https://apps.ecmwf.int/...
    summary:      ERA5 is the fifth generation ECMWF atmospheric reanalysis o...
    title:        ERA5 pressure-levels oper u_component_of_wind 19790101-1979...
    frequency:    day
    CDO:          Climate Data Operators version 2.4.3 (https://mpimet.mpg.de...

In [16]:
u300_chunked = u300.chunk({"time": -1, "lat": -1, "lon": 30})

In [17]:
u300_chunked

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 16802, lat: 90, lon: 360)
Coordinates:
  * time     (time) datetime64[ns] 134kB 1979-01-01T11:00:00 ... 2024-12-31T1...
  * lat      (lat) float64 720B -89.5 -88.5 -87.5 -86.5 ... -3.5 -2.5 -1.5 -0.5
  * lon      (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
    level    int32 4B 300
Data variables:
    u        (time, lat, lon) float32 2GB dask.array<chunksize=(16802, 90, 30), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.4.3 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Tue May 13 00:44:27 2025: cdo mergetime /g/data/if69/ls3248...
    license:      Licence to use Copernicus Products: https://apps.ecmwf.int/...
    summary:      ERA5 is the fifth generation ECMWF atmospheric reanalysis o...
    title:        ERA5 pressure-levels oper u_component_of_wind 19790101-1979...
    frequency:    day
    CDO:          Climate Data Operators version 2.4.3 (https://mpimet.mpg.de...

In [18]:
u300_anoms = doy_anom(u300_chunked)

In [19]:
u300_anoms

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 16802, lon: 360, lat: 90)
Coordinates:
  * time       (time) datetime64[ns] 134kB 1979-01-01T11:00:00 ... 2024-12-31...
    level      (time) int32 67kB 300 300 300 300 300 300 ... 300 300 300 300 300
    dayofyear  (time) int64 134kB 1 2 3 4 5 6 7 ... 360 361 362 363 364 365 366
  * lon        (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
  * lat        (lat) float64 720B -89.5 -88.5 -87.5 -86.5 ... -2.5 -1.5 -0.5
Data variables:
    u          (time, lat, lon) float32 2GB dask.array<chunksize=(366, 90, 30), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.4.3 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Tue May 13 00:44:27 2025: cdo mergetime /g/data/if69/ls3248...
    license:      Licence to use Copernicus Products: https://apps.ecmwf.int/...
    summary:      ERA5 is the fifth generation ECMWF atmospheric reanalysis o...
    title:        ERA5 pressure-levels oper u_component_of_wind 19790101-1979...
    frequency:    day
    CDO:          Climate Data Operators version 2.4.3 (https://mpimet.mpg.de...

In [20]:
u300_anoms = u300_anoms.chunk({"time": -1, "lat": -1, "lon": 30})

In [21]:
u300_anoms

<xarray.Dataset> Size: 2GB
Dimensions:    (time: 16802, lon: 360, lat: 90)
Coordinates:
  * time       (time) datetime64[ns] 134kB 1979-01-01T11:00:00 ... 2024-12-31...
    level      (time) int32 67kB dask.array<chunksize=(16802,), meta=np.ndarray>
    dayofyear  (time) int64 134kB dask.array<chunksize=(16802,), meta=np.ndarray>
  * lon        (lon) float64 3kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
  * lat        (lat) float64 720B -89.5 -88.5 -87.5 -86.5 ... -2.5 -1.5 -0.5
Data variables:
    u          (time, lat, lon) float32 2GB dask.array<chunksize=(16802, 90, 30), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 2.4.3 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Tue May 13 00:44:27 2025: cdo mergetime /g/data/if69/ls3248...
    license:      Licence to use Copernicus Products: https://apps.ecmwf.int/...
    summary:      ERA5 is the fifth generation ECMWF atmospheric reanalysis o...
    title:        ERA5 pressure-levels oper u_component_of_wind 19790101-1979...
    frequency:    day
    CDO:          Climate Data Operators version 2.4.3 (https://mpimet.mpg.de...

In [22]:
u300_encoding = specify_encoding(u300_anoms)

In [23]:
u300_encoding['u']

{'chunks': (16802, 90, 30),
 'compressors': BloscCodec(_tunable_attrs={'typesize'}, typesize=1, cname=<BloscCname.zstd: 'zstd'>, clevel=6, shuffle=<BloscShuffle.bitshuffle: 'bitshuffle'>, blocksize=0),
 'dtype': 'float64'}

In [24]:
write_zarr(
    u300_anoms,
    ERA5_WRITE_PATH,
    'u300_dayofyear_anoms_1979-2024.zarr',
    u300_encoding
)